# AI-LUT Training (REQ-004)

Ingests `unified_log.txt` from the camera SD card, trains a
`DecisionTreeClassifier(max_depth=4)` predicting ISO from LightLevel, applies the
finalized integer ETTR/ALO/HTP thresholds and the scene WB table, and exports a
deterministic `unified.tbl`. Produces byte-identical output to
`lut_training_multi_param.py` for the same input.


In [ ]:
# Cell 1: Upload unified_log.txt from your SD card (A:/ML/logs/unified_log.txt)
from google.colab import files
uploaded = files.upload()
LOG_PATH = list(uploaded.keys())[0] if uploaded else "unified_log.txt"
print("Using log:", LOG_PATH)


In [ ]:
# Cell 2: Parse the key=value log (entries separated by "---") into a DataFrame
import pandas as pd

def parse_log(path):
    with open(path, "r", encoding="utf-8") as fh:
        raw = fh.read()
    records = []
    for block in raw.split("---"):
        block = block.strip()
        if not block:
            continue
        entry = {}
        for line in block.splitlines():
            if "=" in line:
                key, value = line.split("=", 1)
                entry[key.strip()] = value.strip()
        if entry:
            records.append(entry)
    if not records:
        raise ValueError(f"No log entries found in {path!r}; cannot train on an empty log.")
    return pd.DataFrame(records)

df = parse_log(LOG_PATH)
df.head()


In [ ]:
# Cell 3: Feature engineering -- LightLevel int, ISO int, Scene str
if "LightLevel" not in df.columns or "ISO" not in df.columns:
    raise ValueError("Log is missing required 'LightLevel' or 'ISO' column.")
df["LightLevel"] = pd.to_numeric(df["LightLevel"], errors="coerce")
df["ISO"] = pd.to_numeric(df["ISO"], errors="coerce")
if "Scene" not in df.columns:
    df["Scene"] = "unknown"
df["Scene"] = df["Scene"].fillna("unknown").astype(str)
bad = df[df["LightLevel"].isna() | df["ISO"].isna()]
for _ in bad.index:
    print("WARNING: skipping row with missing LightLevel or ISO")
df = df.dropna(subset=["LightLevel", "ISO"]).reset_index(drop=True)
df["LightLevel"] = df["LightLevel"].astype(int)
df["ISO"] = df["ISO"].astype(int)
df[["Scene", "LightLevel", "ISO"]]


In [ ]:
# Cell 4: Train DecisionTreeClassifier(max_depth=4) on [LightLevel] -> ISO
from sklearn.tree import DecisionTreeClassifier
RANDOM_STATE = 42
if len(df) < 5:
    print(f"WARNING: only {len(df)} training samples (<5); training anyway.")
clf = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
clf.fit(df[["LightLevel"]], df["ISO"])
print("Trained on", len(df), "samples; classes:", list(clf.classes_))


In [ ]:
# Cell 5: Generate LUT rows using integer threshold rules + scene WB table
WB_TABLE = {
    "daylight": (120, 100, 90),
    "shade": (110, 100, 105),
    "tungsten": (150, 100, 70),
    "lowlight": (130, 100, 80),
    "unknown": (100, 100, 100),
}

def ettr_for(light):
    if light > 200:
        return "reduce_shutter"
    if light < 30:
        return "increase_shutter"
    return "keep_shutter"

def alo_for(light):
    if light < 20:
        return "shadow_boost"
    if light < 50:
        return "shadow_lift"
    return "neutral"

def htp_for(light):
    if light < 30:
        return "priority_on"
    return "priority_off"

def wb_for(scene):
    r, g, b = WB_TABLE.get(scene, (100, 100, 100))
    return f"R{r},G{g},B{b}"

# sort by Scene then LightLevel (tuples sort lexicographically), one row per pair
pairs = sorted(set(zip(df["Scene"], df["LightLevel"])))
rows = []
for scene, light in pairs:
    light = int(light)
    iso = int(clf.predict(pd.DataFrame({"LightLevel": [light]}))[0])
    rows.append(
        f"{scene}|{light}|{ettr_for(light)}|{alo_for(light)}|"
        f"{htp_for(light)}|{iso}|{wb_for(scene)}"
    )
rows


In [ ]:
# Cell 6: Write unified.tbl (UTF-8, no BOM, LF, sorted) and download
HEADER = "# Scene|LightLevel|ETTR|ALO|HTP|ISO|WB"
LUT_PATH = "unified.tbl"
with open(LUT_PATH, "w", encoding="utf-8", newline="\n") as fh:
    fh.write(HEADER + "\n")
    for row in rows:
        fh.write(row + "\n")
print("Wrote", LUT_PATH, "with", len(rows), "rows")

from google.colab import files
files.download(LUT_PATH)
